# 17th ACM Conference on Bioinformatics, Computational Biology, and Health Informatics (ACM-BCB 2026), University of Calabria.

## T1: Spatial transcriptomics meets advanced image analysis: AI-driven integration of spatial omics data
**Authors:** Sara Terzoli & Matteo Bonfanti \
*National Facility for Data Handling and Analysis, Human Technopole, Milan, Italy.*

In this hands-on tutorial, we present a Visium HD analysis workflow integrating spatial transcriptomics data with image-based cell segmentation to infer cell identities and investigate their spatial distribution within the tissue.

The notebook is organized in sequential sections so participants can run one block at a time during the live tutorial.

## Biological question and dataset used in this tutorial

In this tutorial we work with **Visium HD spatial transcriptomics data from human colorectal cancer (CRC)**. The analysis is inspired by the Nature Genetics study *High-definition spatial transcriptomic profiling of immune cell populations in colorectal cancer* (DOI: `10.1038/s41588-025-02193-3`). That work uses Visium HD to study FFPE human CRC samples and to resolve how tumour, stromal and immune populations are spatially organized within the tumour microenvironment.

For the hands-on part, we use the public **10x Genomics Human Colorectal Cancer (FFPE) Visium HD dataset**, specifically **sample P5**. 

The first step of the analysis is quality control (QC).
We assess whether each spatial bin contains:
- a sufficient number of detected molecules and genes,
- evaluate mitochondrial transcript abundance, 
- identify potential low-quality regions across the tissue. 
To facilitate this assessment, QC metrics are projected back onto the tissue section, allowing us to visualize their spatial distribution and detect localized technical artifacts.

Reference links:
- Nature Genetics paper: https://www.nature.com/articles/s41588-025-02193-3
- 10x Genomics dataset page: https://www.10xgenomics.com/platforms/visium/product-family/dataset-human-crc


## 1. Environment

### Analysis setup

Spatial transcriptomics analyses combine several data layers: the count matrix, spot coordinates, tissue images and metadata. For this reason, reproducibility depends not only on the code, but also on the software versions and plotting backends. In this section we import the main libraries and define paths and thresholds once, so the rest of the notebook can be re-run in a controlled way.

Key objects used in this tutorial:

- **SpatialData (`sdata`)**: container that keeps together images, shapes, coordinate systems and AnnData tables.
- **AnnData (`adata`)**: table-like object containing the gene expression matrix and per-bin/per-gene annotations.
- **Shapes**: geometries representing Visium HD square bins.


### 1.1 Import Libraries

The main libraries used in this tutorial are:

- **Scanpy**: for handling AnnData objects, calculating quality-control metrics, and performing downstream single-cell analyses.
- **SpatialData** for managing, integrating, and visualizing spatial omics datasets, including images, segmentation masks, points, shapes, and spatially resolved molecular measurements within a unified data model.
- **SciPy Sparse**: for efficiently storing and loading segmentation masks in sparse `.npz` format.
- **Matplotlib** and **Seaborn**: for data visualization and quality-control plots.

In [ ]:
# Standard library utilities for paths, memory, randomness, and warnings.
from pathlib import Path
import random
from datetime import datetime

# Core scientific Python libraries for arrays, dataframes, sparse matrices, and plotting.
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.colors import Normalize
import numpy as np
import pandas as pd
import scipy.sparse as sp
import seaborn as sns

# Scanpy and spatial data utilities for single-cell/spatial omics processing.
import scanpy as sc
import spatialdata as sd
import spatialdata_plot

# Deep learning / probabilistic modeling support used by Cell2location.
import torch

### 1.2 Settings

In [ ]:
# Use the same random seed every time so stochastic steps are reproducible.
seed = 42
np.random.seed(seed)
random.seed(seed)

In [ ]:
# Set global plotting defaults.
sc.set_figure_params(facecolor="white", figsize=(8, 8))
sc.settings.verbosity = 3
sns.set_style("whitegrid")
mpl.rcParams["figure.dpi"] = 100

### 1.3 Files and parameters

In [ ]:
# Tutorial dataset: P5 (10x Genomics Visium HD colorectal cancer sample).
# Update the path if needed.
samples = [
    {"path": Path("/workspaces/data/P5_full_slide.zarr"), "id": "P5"},
]

In [ ]:
sample_path = samples[0]["path"]
sample_id = samples[0]["id"]

print(sample_path)

In [ ]:
results_dir = Path("/workspaces/results/")
results_dir.mkdir(parents=True, exist_ok=True)

### 1.4 Start computations

In [ ]:
print(datetime.now())

## 2. Data Load 

### Theory: SpatialData versus AnnData

Visium HD data are more complex than a standard single-cell count matrix. The experiment contains both molecular information and spatial context. `SpatialData` is useful because it stores all these layers in one object:

- images: the high-resolution tissue image;
- shapes: the square bins generated by Space Ranger;
- tables: AnnData objects containing counts and metadata;
- coordinate systems: the alignment information linking images, shapes, and tables.

The SpatialData object contains three binning resolutions generated by Space Ranger (square_016um, square_008um, and square_002um). In this tutorial, we focus on the square_016um resolution, which provides a good balance between spatial detail and transcript counts per bin, making it particularly suitable for quality control, filtering, and downstream analyses.

In this tutorial we extract the square_016um table as an AnnData object for QC and filtering, but we keep the full sdata object for plotting with the spatialdata_plot function.

**Reference:** Marconato et al., Nature Methods 2024; SpatialData documentation: https://spatialdata.scverse.org

### 2.1 Load the SpatialData object from an existing Zarr store.

In [ ]:
# Load the SpatialData object
sdata = sd.read_zarr(sample_path)

In [ ]:
sdata

In [ ]:
table_key = "square_016um"
shape_key = "P5_square_016um"

adata = sdata.tables[table_key]
adata.obs["sample"] = sample_id

print(f"Using SpatialData table: {table_key}")
print(f"Using SpatialData shapes: {shape_key}")

#### Understanding the AnnData object

This dataset contains **136,035 spatial bins** (`n_obs`) and **18,085 genes** (`n_vars`).

- `obs`: metadata for each spatial bin, including its position on the Visium HD array (`array_row`, `array_col`) and whether it falls within the tissue (`in_tissue`).
- `var`: metadata for each gene, such as gene identifiers and feature annotations.
- `obsm["spatial"]`: spatial coordinates used to map bins back onto the tissue image.
- `uns`: additional unstructured metadata, including information linking the table to the SpatialData object.

> **Rows = spatial bins (`n_obs`)**  
> **Columns = genes (`n_vars`)**

In [ ]:
adata

## 3. Pre-filter quality control

### Theory: quality control before filtering

Quality control checks whether each spatial bin contains enough biological signal to be useful for downstream analysis. Here we calculate common QC metrics:

- `total_counts`: total number of UMIs detected in each bin;
- `n_genes_by_counts`: number of detected genes per bin;
- `pct_counts_mt`: percentage of counts assigned to mitochondrial genes.

Low counts or very few detected genes can indicate low-quality bins. A high mitochondrial percentage can indicate damaged tissue or stressed cells. We first annotate potential outliers, then inspect their spatial distribution before removing them.

The spatial QC plots are generated using `sdata.pl.render_shapes(...)`.

In [ ]:
total_spots = adata.n_obs
n_in_tissue = int((adata.obs["in_tissue"] == 1).sum())
print(f"Total spots: {total_spots}")
print(f"In-tissue spots: {n_in_tissue}")

In [ ]:
# Count detected genes and total UMIs per spot.
genes_per_spot = np.asarray((adata.X > 0).sum(axis=1)).ravel()
umi_per_spot = np.asarray(adata.X.sum(axis=1)).ravel()
mean_genes_per_spot = float(genes_per_spot.mean())
mean_umi_per_spot = float(umi_per_spot.mean())

print(f"Mean genes per spot: {mean_genes_per_spot:.2f}")
print(f"Mean UMI per spot: {mean_umi_per_spot:.2f}")

In [ ]:
# Flag mitochondrial genes before calculating QC metrics.
adata.var["mt"] = adata.var_names.str.upper().str.startswith("MT-")

In [ ]:
# Calculate standard Scanpy QC metrics and percentages for the flagged gene groups.
sc.pp.calculate_qc_metrics(
    adata,
    qc_vars=["mt"],
    inplace=True,
    percent_top=[20],
    log1p=True,
)

In [ ]:
# QC thresholds used in filtering section.
min_counts = 200
min_genes = 100
max_mt = 60
min_cells = 3

In [ ]:
# Plot QC metric distributions before filtering.

num_panels = 3 if "pct_counts_mt" in adata.obs.columns else 2
fig, axs = plt.subplots(1, num_panels, figsize=(6 * num_panels, 5))

sns.histplot(adata.obs["total_counts"], kde=True, color="skyblue", ax=axs[0])
axs[0].set_title(f"{sample_id} - Total Counts per Spot")
axs[0].set_xlabel("Total counts")
axs[0].set_ylabel("Number of spots")
axs[0].axvline(x=min_counts, color="red", linestyle="--", label=f"min_counts={min_counts}")
axs[0].legend()

sns.histplot(adata.obs["n_genes_by_counts"], kde=True, bins=60, color="lightgreen", ax=axs[1])
axs[1].set_title(f"{sample_id} - Genes Detected per Spot")
axs[1].set_xlabel("Number of genes")
axs[1].set_ylabel("Number of spots")
axs[1].axvline(x=min_genes, color="red", linestyle="--", label=f"min_genes={min_genes}")
axs[1].legend()

if num_panels == 3:
    sns.histplot(adata.obs["pct_counts_mt"], kde=True, bins=60, color="salmon", ax=axs[2])
    axs[2].set_title(f"{sample_id} - Percent Mitochondrial Counts")
    axs[2].set_xlabel("% mt counts")
    axs[2].set_ylabel("Number of spots")
    axs[2].axvline(x=max_mt, color="red", linestyle="--", label=f"max_mt={max_mt}")
    axs[2].legend()

plt.tight_layout()
plt.show()

In [ ]:
# Outliers are identified using the thresholds defined at the beginning of the notebook.
adata.obs["qc_lib_size"] = adata.obs["total_counts"] < min_counts
adata.obs["qc_detected"] = adata.obs["n_genes_by_counts"] < min_genes

if "pct_counts_mt" in adata.obs.columns:
    adata.obs["qc_mito"] = adata.obs["pct_counts_mt"] > max_mt
else:
    adata.obs["qc_mito"] = False

# Low-complexity spots are recorded above but not removed automatically here.
adata.obs["global_outliers"] = (
    adata.obs["qc_lib_size"]
    | adata.obs["qc_detected"]
    | adata.obs["qc_mito"]
)

print("Global outlier counts:")
print(adata.obs["global_outliers"].value_counts())

#### Spatial QC visualization with SpatialData shapes

For Visium HD square-bin data, each bin is represented as a spatial shape. After updating sdata.tables[table_key] with the QC-annotated AnnData table, we visualize the main QC metrics in their spatial context using spatialdata_plot, including library size (total_counts), detected genes (n_genes_by_counts), mitochondrial content (pct_counts_mt), and the global QC outlier status (global_outliers).


In [ ]:
sdata.pl.render_images(
    "P5_hires_image",
    grayscale=True,
    cmap="grey",
).pl.show(
    coordinate_systems=sample_id,
)

In [ ]:
sdata_crop = sdata.query.bounding_box(
    axes=("x", "y"),
    min_coordinate=[41000, 37000],
    max_coordinate=[66000, 65000],
    target_coordinate_system="P5",
)

In [ ]:
sample_np = adata.obs["total_counts"].to_numpy()
vmin = np.percentile(sample_np, 1)
vmax = np.percentile(sample_np, 99)
norm = Normalize(vmin=vmin, vmax=vmax)

sdata_crop.pl.render_images(
    "P5_hires_image",
    grayscale=True,
    cmap="grey",
).pl.render_shapes(
    shape_key,
    color="total_counts",
    cmap="viridis",
    fill_alpha=0.5,
    outline_width=0,
    norm=norm
).pl.show(
    coordinate_systems=sample_id,
    dpi=100
)


In [ ]:
sample_np = adata.obs["pct_counts_mt"].to_numpy()
vmin = np.percentile(sample_np, 1)
vmax = np.percentile(sample_np, 99)
norm = Normalize(vmin=vmin, vmax=vmax)

sdata_crop.pl.render_images(
    "P5_hires_image",
    grayscale=True,
    cmap="grey",
).pl.render_shapes(
    shape_key,
    color="pct_counts_mt",
    cmap="viridis",
    fill_alpha=0.5,
    outline_width=0,
    norm=norm
).pl.show(
    coordinate_systems=sample_id,
    dpi=100
)


In [ ]:
sdata_crop.pl.render_images(
    "P5_hires_image",
    grayscale=True,
    cmap="grey",
).pl.render_shapes(
    shape_key,
    color="global_outliers",
    fill_alpha=0.5,
    outline_width=0,
    cmap="Reds"
).pl.show(
    coordinate_systems=sample_id,
    dpi=100
)


In [ ]:
del sdata_crop

In [ ]:
# Save a one-row CSV summary and the AnnData object with QC annotations.
summary_dict = {
    "Sample": sample_id,
    "Total spots": total_spots,
    "In-tissue spots": n_in_tissue,
    "Mean genes per spot": mean_genes_per_spot,
    "Mean UMI per spot": mean_umi_per_spot,
    "Global outliers (True)": int(adata.obs["global_outliers"].sum()),
    "Global outliers (False)": int((~adata.obs["global_outliers"]).sum()),
}

if "pct_counts_mt" in adata.obs.columns:
    summary_dict["Mean % MT"] = adata.obs["pct_counts_mt"].mean()
    summary_dict["Median % MT"] = adata.obs["pct_counts_mt"].median()


In [ ]:
print("\n" + "=" * 50)
print(f"QC SUMMARY - {sample_id}")
print("=" * 50)

for k, v in summary_dict.items():
    if isinstance(v, float):
        print(f"{k}: {v:.2f}")
    else:
        print(f"{k}: {v}")

print("=" * 50)

## 4. Filtering

### Theory: applying QC-based filters

We next apply QC-based filtering to remove low-quality spatial bins. The filters are based on the metrics computed in the previous section and are designed to retain bins with sufficient molecular information.

After bin filtering, genes detected in too few retained bins are also removed. The resulting dataset is used for downstream analyses such as deconvolution.

In [ ]:
print(f"Shape before filtering: {adata.shape}")

# Mark spots to discard based on the QC flags generated above.
n_filtered_spots = int(adata.obs["global_outliers"].sum())
print(f"Number of spots filtered out: {n_filtered_spots}")

In [ ]:
# Keep high-quality spots and continue using `adata_qc` for the filtered object.
adata = adata[~adata.obs["global_outliers"]].copy()

In [ ]:
# Remove genes detected in fewer than `min_cells` retained spots.
gene_nspots = np.asarray((adata.X > 0).sum(axis=0)).ravel()
adata = adata[:, gene_nspots >= min_cells].copy()

In [ ]:
sdata.tables["square_016um"] = adata

In [ ]:
# Save filtered object
filtered_h5ad = results_dir / f"{sample_id}_016um_filtered.h5ad"
adata.write(filtered_h5ad)

In [ ]:
print(f"Shape after filtering: {adata.shape}")

In [ ]:
# Plot QC metric distributions after filtering.

num_panels = 3
fig, axs = plt.subplots(1, num_panels, figsize=(6 * num_panels, 5))

sns.histplot(
    adata.obs["total_counts"],
    kde=True,
    color="skyblue",
    ax=axs[0]
)
axs[0].set_title(f"{sample_id} - Total Counts per Spot")
axs[0].set_xlabel("Total counts")
axs[0].set_ylabel("Number of spots")

sns.histplot(
    adata.obs["n_genes_by_counts"],
    kde=True,
    bins=60,
    color="lightgreen",
    ax=axs[1]
)
axs[1].set_title(f"{sample_id} - Genes Detected per Spot")
axs[1].set_xlabel("Number of genes")
axs[1].set_ylabel("Number of spots")

sns.histplot(
    adata.obs["pct_counts_mt"],
    kde=True,
    bins=60,
    color="salmon",
    ax=axs[2]
)
axs[2].set_title(f"{sample_id} - Percent Mitochondrial Counts")
axs[2].set_xlabel("% mt counts")
axs[2].set_ylabel("Number of spots")

plt.tight_layout()
plt.show()


#### Post-filter spatial check

After filtering, we repeat the spatial visualization to confirm that retained bins still cover the expected tissue region and that removed bins do not create unexpected holes or spatial biases.


In [ ]:
sdata_crop = sdata.query.bounding_box(
    axes=("x", "y"),
    min_coordinate=[45000, 45000],
    max_coordinate=[55000, 55000],
    target_coordinate_system="P5",
)

In [ ]:
sample_np = adata.obs["total_counts"].to_numpy()
vmin = np.percentile(sample_np, 1)
vmax = np.percentile(sample_np, 99)
norm = Normalize(vmin=vmin, vmax=vmax)

sdata_crop.pl.render_images(
    "P5_hires_image",
    grayscale=True,
    cmap="grey",
).pl.render_shapes(
    shape_key,
    color="total_counts",
    cmap="viridis",
    fill_alpha=0.5,
    outline_width=0,
    norm=norm
).pl.show(
    coordinate_systems=sample_id,
    dpi=100
)


In [ ]:
sample_np = adata.obs["pct_counts_mt"].to_numpy()
vmin = np.percentile(sample_np, 1)
vmax = np.percentile(sample_np, 99)
norm = Normalize(vmin=vmin, vmax=vmax)

sdata_crop.pl.render_images(
    "P5_hires_image",
    grayscale=True,
    cmap="grey",
).pl.render_shapes(
    shape_key,
    color="pct_counts_mt",
    cmap="viridis",
    fill_alpha=0.5,
    outline_width=0,
    norm=norm
).pl.show(
    coordinate_systems=sample_id,
    dpi=100
)


## Summary

In this tutorial we:
- Loaded a Visium HD dataset as a **SpatialData** object.
- Extracted the **16 µm AnnData table** for downstream analysis.
- Calculated key **QC metrics** (`total_counts`, `n_genes_by_counts`, `pct_counts_mt`).
- Visualized QC metrics in their spatial context using **SpatialData shapes**.
- Applied **QC-based filtering** to remove low-quality spatial bins.
- Removed genes detected in too few retained bins.
- Re-evaluated the dataset after filtering through spatial QC visualization.

### Key takeaways

- **SpatialData (`sdata`)** stores images, shapes, tables, and coordinate systems in a single framework.
- **AnnData (`adata`)** contains the gene expression matrix and associated metadata.